# Feature Extraction with Missing Feature Handling — Google Drive Edition

This notebook walks a folder tree of raw student submissions in Google Drive, extracts modality-aware features, and writes a single feature DataFrame to `MEng Path Assignments/Regression Analysis Datasets/feature_extraction_output.csv`.

**Inputs**: folders of raw files organized by student. Supported extensions:

| Extension | Modality | Treatment |
|---|---|---|
| `.py` | code | Read as source code |
| `.ipynb` | code | Code cells concatenated; markdown cells dropped |
| `.txt`, `.md` | text | Read as plain text |
| `.vtt`, `.srt` | oral_video | Timestamps stripped; only spoken text retained |

**Score fields** (`ai_score_norm`, `human_score_norm`, `semantic_entropy`, `ai_confidence`, `disagreement`) are intentionally left blank in this run. They will be merged in downstream from the LLM scoring pipeline.

**Outputs**: a single CSV at `MEng Path Assignments/Regression Analysis Datasets/feature_extraction_output.csv` with one row per (student × assignment × question) record and the full canonical feature schema with explicit `*_missing` indicator columns.

## 1. Mount Google Drive

Run once per Colab session. You'll be prompted to authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

**Edit `INPUT_DIR_RELATIVE` below** to point at the folder in your Drive that contains the per-student submission folders. The path is relative to `/content/drive/MyDrive/`, which is your Drive root once mounted.

Examples:
- `MEng Path Assignments/Submissions`
- `MEng Path Assignments/Raw Data/Spring 2026`
- `MEng Path Assignments/Submissions/Mock Interviews`

The output path is fixed to `MEng Path Assignments/Regression Analysis Datasets/feature_extraction_output.csv` per the project convention.

In [ ]:
from pathlib import Path

# ============================================================
# EDIT THIS PATH — relative to /content/drive/MyDrive/
# ============================================================
INPUT_DIR_RELATIVE = "MEng Path Assignments/Submissions"
# ============================================================

DRIVE_ROOT      = Path("/content/drive/MyDrive")
INPUT_DIR       = DRIVE_ROOT / INPUT_DIR_RELATIVE
OUTPUT_DIR      = DRIVE_ROOT / "MEng Path Assignments" / "Regression Analysis Datasets"
OUTPUT_FILENAME = "feature_extraction_output.csv"
OUTPUT_PATH     = OUTPUT_DIR / OUTPUT_FILENAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_DIR.exists(), (
    f"Input folder not found at:\n  {INPUT_DIR}\n"
    f"Edit INPUT_DIR_RELATIVE above. Make sure Drive is mounted "
    f"and the folder name matches exactly (case-sensitive)."
)

print(f"Input dir : {INPUT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Output file: {OUTPUT_PATH}")
print()
print(f"Top-level contents of input folder ({sum(1 for _ in INPUT_DIR.iterdir())} entries):")
for p in sorted(INPUT_DIR.iterdir())[:25]:
    kind = "📁" if p.is_dir() else "📄"
    print(f"  {kind} {p.name}")

## 3. Imports

In [ ]:
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Tuple
import ast
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

## 4. Data structure

`FeatureRecord` is the canonical row format — one record per (student × assignment × question). Score fields are typed `Optional` and default to `None` because this run does features only; scores will be merged in downstream.

In [ ]:
@dataclass
class FeatureRecord:
    student_id: str
    assignment_id: str
    question_id: str
    modality: str
    rubric_type: str
    features: Dict[str, Any] = None
    ai_score_norm: Optional[float] = None
    human_score_norm: Optional[float] = None
    semantic_entropy: Optional[float] = None
    ai_confidence: Optional[float] = None

    def disagreement(self) -> Optional[float]:
        if self.human_score_norm is None or self.ai_score_norm is None:
            return None
        return abs(self.ai_score_norm - self.human_score_norm)

## 5. Canonical feature schema

Every record gets the same feature columns. Features that don't apply to a given modality are left missing and paired with `*_missing` indicator columns so downstream regressions can model structural missingness explicitly rather than imputing it away.

In [ ]:
CANONICAL_FEATURES = {
    # Text
    "response_length_words": np.nan,
    "response_length_sentences": np.nan,
    "avg_sentence_length": np.nan,
    "lexical_diversity_ttr": np.nan,
    "has_evidence_marker": np.nan,
    "estimated_num_claims": np.nan,

    # Code
    "code_length_lines": np.nan,
    "code_length_chars": np.nan,
    "num_functions": np.nan,
    "syntax_error": np.nan,
    "cyclomatic_complexity_proxy": np.nan,

    # Visualization / EDA
    "chart_type": "not_applicable",
    "has_xlabel": np.nan,
    "has_ylabel": np.nan,
    "has_title": np.nan,
    "num_detected_transformations": np.nan,

    # Oral / video
    "transcript_length_words": np.nan,
    "speaking_tempo_wpm": np.nan,
    "filler_word_count": np.nan,
    "filler_word_rate": np.nan,
    "star_situation_present": np.nan,
    "star_task_present": np.nan,
    "star_action_present": np.nan,
    "star_result_present": np.nan,
    "star_completeness_score": np.nan,
}


def is_missing_value(value: Any) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def with_schema(features: Dict[str, Any]) -> Dict[str, Any]:
    """Return a full feature dictionary with explicit missingness flags."""
    merged = dict(CANONICAL_FEATURES)
    merged.update(features)
    for key in list(CANONICAL_FEATURES.keys()):
        value = merged.get(key, np.nan)
        if isinstance(CANONICAL_FEATURES[key], str):
            merged[f"{key}_missing"] = int(value in [None, "missing", "not_applicable"])
        else:
            merged[f"{key}_missing"] = int(is_missing_value(value))
    return merged

## 6. Text feature extraction

In [ ]:
def simple_sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]


def tokenize(text: str) -> List[str]:
    return re.findall(r"\b\w+\b", text.lower())


def extract_text_features(text: Optional[str]) -> Dict[str, Any]:
    if not text:
        return with_schema({})
    tokens = tokenize(text)
    sentences = simple_sentences(text)
    num_words = len(tokens)
    num_sentences = len(sentences)
    unique_words = len(set(tokens))
    evidence_markers = [
        "because", "for example", "for instance", "according to",
        "the data shows", "this suggests", "evidence", "figure", "table"
    ]
    features = {
        "response_length_words": num_words,
        "response_length_sentences": num_sentences,
        "avg_sentence_length": num_words / max(num_sentences, 1),
        "lexical_diversity_ttr": unique_words / max(num_words, 1),
        "has_evidence_marker": int(any(marker in text.lower() for marker in evidence_markers)),
        "estimated_num_claims": sum(1 for s in sentences if len(tokenize(s)) >= 5),
    }
    return with_schema(features)

## 7. Code feature extraction

Test-runner integration was removed for this pipeline because we don't have a sandboxed test harness for the submitted student code in Colab. Test-related fields (`passed_tests`, `total_tests`, `test_pass_rate`) were dropped from the canonical schema accordingly. If you add a sandboxed runner later, re-introduce them in `CANONICAL_FEATURES` above.

In [ ]:
def extract_code_features(code_text: Optional[str]) -> Dict[str, Any]:
    if not code_text:
        return with_schema({})
    features = {
        "code_length_lines": len(code_text.splitlines()),
        "code_length_chars": len(code_text),
        "num_functions": 0,
        "syntax_error": 0,
        "cyclomatic_complexity_proxy": 0,
    }
    try:
        tree = ast.parse(code_text)
        features["num_functions"] = sum(isinstance(n, ast.FunctionDef) for n in ast.walk(tree))
        branch_nodes = (
            ast.If, ast.For, ast.While, ast.Try, ast.ExceptHandler,
            ast.BoolOp, ast.IfExp, ast.With, ast.Assert,
        )
        features["cyclomatic_complexity_proxy"] = 1 + sum(
            isinstance(n, branch_nodes) for n in ast.walk(tree)
        )
    except SyntaxError:
        features["syntax_error"] = 1
    return with_schema(features)

## 8. EDA / visualization feature extraction

EDA features are extracted opportunistically from any code submission that contains matplotlib / seaborn / pandas-plot calls. A pure-text response with no plotting code will produce all-missing EDA features, which is the intended behavior.

In [ ]:
def extract_visualization_features(
    code_text: Optional[str],
    explanation: Optional[str] = None,
) -> Dict[str, Any]:
    features = {}
    if code_text:
        lower_code = code_text.lower()
        chart_type = "unknown"
        chart_patterns = {
            "line":      ["plt.plot", ".plot("],
            "bar":       ["plt.bar", "kind='bar'", 'kind="bar"'],
            "scatter":   ["plt.scatter", "kind='scatter'", 'kind="scatter"'],
            "histogram": ["plt.hist", "kind='hist'", 'kind="hist"'],
            "heatmap":   ["heatmap"],
            "boxplot":   ["boxplot", "plt.boxplot"],
        }
        for ctype, patterns in chart_patterns.items():
            if any(p in lower_code for p in patterns):
                chart_type = ctype
                break
        transform_markers = [
            "groupby", "merge", "join", "filter",
            "dropna", "fillna", "sort_values", "pivot",
        ]
        features.update({
            "chart_type": chart_type,
            "has_xlabel": int("xlabel" in lower_code or ".set_xlabel" in lower_code),
            "has_ylabel": int("ylabel" in lower_code or ".set_ylabel" in lower_code),
            "has_title":  int("title"  in lower_code or ".set_title"  in lower_code),
            "num_detected_transformations": sum(m in lower_code for m in transform_markers),
        })
    if explanation:
        text_features = extract_text_features(explanation)
        for k, v in text_features.items():
            if k in CANONICAL_FEATURES or k.endswith("_missing"):
                features[f"viz_explanation_{k}"] = v
    return with_schema(features)

## 9. Oral / video transcript feature extraction

Speaking tempo (`speaking_tempo_wpm`) requires a duration in seconds. Since we're parsing `.vtt` / `.srt` files directly, the duration is recovered from the **last cue's end timestamp** in the file rather than asking the user to supply it. If the timestamp parser fails (malformed cue), tempo is left missing.

In [ ]:
FILLER_WORDS = {"um", "uh", "erm", "like", "you know", "sort of", "kind of"}


def count_filler_words(transcript: str) -> int:
    text = transcript.lower()
    return sum(
        len(re.findall(rf"\b{re.escape(filler)}\b", text))
        for filler in FILLER_WORDS
    )


def extract_oral_features(
    transcript: Optional[str],
    duration_seconds: Optional[float] = None,
) -> Dict[str, Any]:
    if not transcript:
        return with_schema({})
    tokens = tokenize(transcript)
    num_words = len(tokens)
    filler_count = count_filler_words(transcript)
    tempo_wpm = np.nan
    if duration_seconds and duration_seconds > 0:
        tempo_wpm = num_words / (duration_seconds / 60)
    lower = transcript.lower()
    star_markers = {
        "situation": ["situation", "context", "when", "during"],
        "task":      ["task", "goal", "responsibility", "needed to"],
        "action":    ["i did", "we did", "i used", "i implemented", "i analyzed"],
        "result":    ["result", "outcome", "impact", "improved", "learned"],
    }
    star_presence = {
        f"star_{key}_present": int(any(marker in lower for marker in markers))
        for key, markers in star_markers.items()
    }
    features = {
        "transcript_length_words": num_words,
        "speaking_tempo_wpm": tempo_wpm,
        "filler_word_count": filler_count,
        "filler_word_rate": filler_count / max(num_words, 1),
        **star_presence,
        "star_completeness_score": sum(star_presence.values()) / 4,
    }
    return with_schema(features)

## 10. Modality-aware feature assembly

In [ ]:
def extract_features_for_question(
    modality: str,
    text: Optional[str] = None,
    code_text: Optional[str] = None,
    explanation: Optional[str] = None,
    transcript: Optional[str] = None,
    duration_seconds: Optional[float] = None,
) -> Dict[str, Any]:
    if modality == "text":
        return extract_text_features(text)
    if modality == "code":
        return extract_code_features(code_text)
    if modality == "eda":
        return extract_visualization_features(
            code_text=code_text,
            explanation=explanation or text,
        )
    if modality == "oral_video":
        return extract_oral_features(
            transcript=transcript or text,
            duration_seconds=duration_seconds,
        )
    return with_schema({})


def records_to_dataframe(records: List[FeatureRecord]) -> pd.DataFrame:
    rows = []
    for r in records:
        row = {
            "student_id": r.student_id,
            "assignment_id": r.assignment_id,
            "question_id": r.question_id,
            "modality": r.modality,
            "rubric_type": r.rubric_type,
            "ai_score_norm": r.ai_score_norm,
            "human_score_norm": r.human_score_norm,
            "semantic_entropy": r.semantic_entropy,
            "ai_confidence": r.ai_confidence,
            "disagreement": r.disagreement(),
        }
        row.update(with_schema(r.features or {}))
        rows.append(row)
    return pd.DataFrame(rows)

## 11. File-content readers

Each file extension gets a small helper that returns clean content suitable for the feature extractors. Treatment:

- `.py` — read as-is.
- `.ipynb` — code cells are concatenated with `\n\n# --- cell break ---\n\n` between them; markdown cells are dropped, so what reaches the AST parser is exactly what the student wrote as Python.
- `.txt`, `.md` — read as-is.
- `.vtt`, `.srt` — timestamp lines, cue numbers, and `WEBVTT` headers are stripped; only spoken text remains. Total duration in seconds is recovered from the last cue's end timestamp when available.

In [ ]:
def read_python_file(path: Path) -> str:
    return path.read_text(errors="ignore")


def read_ipynb_file(path: Path) -> str:
    """Concatenate code cells from an .ipynb. Drops markdown cells."""
    try:
        nb = json.loads(path.read_text(errors="ignore"))
    except json.JSONDecodeError:
        return ""
    code_chunks = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)
        if src.strip():
            code_chunks.append(src)
    return "\n\n# --- cell break ---\n\n".join(code_chunks)


def read_text_file(path: Path) -> str:
    return path.read_text(errors="ignore")


_VTT_TIMESTAMP = re.compile(
    r"(\d{1,2}):(\d{2}):(\d{2})[.,](\d{1,3})"  # HH:MM:SS.mmm or HH:MM:SS,mmm (SRT)
)


def read_vtt_or_srt(path: Path) -> Tuple[str, Optional[float]]:
    """Return (clean transcript text, total duration in seconds or None)."""
    raw = path.read_text(errors="ignore")
    lines = raw.splitlines()
    spoken = []
    last_end_seconds: Optional[float] = None

    for line in lines:
        stripped = line.strip()
        # Skip the WEBVTT header
        if stripped.upper().startswith("WEBVTT"):
            continue
        # Skip cue numbers (a bare integer line, common in .srt)
        if stripped.isdigit():
            continue
        # Cue timing line — parse end time and skip the line itself
        if "-->" in stripped:
            parts = stripped.split("-->")
            if len(parts) == 2:
                m = _VTT_TIMESTAMP.search(parts[1])
                if m:
                    h, mn, s, ms = m.groups()
                    last_end_seconds = (
                        int(h) * 3600 + int(mn) * 60 + int(s) + int(ms) / 1000.0
                    )
            continue
        # Skip blank lines
        if not stripped:
            continue
        # Otherwise it's spoken text
        spoken.append(stripped)

    return " ".join(spoken), last_end_seconds

## 12. Folder walker — student / assignment / question parsing

The walker auto-detects which of three common layouts you have, based on the depth at which submission files first appear:

| Inferred layout | Example | Parsing |
|---|---|---|
| **flat** | `INPUT/S001_A001_Q1.py` | All three IDs from filename, split on underscore |
| **per_student** | `INPUT/S001/A001_Q1.py` | Student from folder; assignment + question from filename |
| **per_student_per_assignment** | `INPUT/S001/A001/Q1.py` | Student + assignment from folders; question from filename |

The detector prints what it inferred so you can verify before the run.

If your layout is different, edit the `parse_ids_from_path` function below — it's intentionally short and the only place you need to change.

In [ ]:
SUPPORTED_EXTENSIONS = {
    ".py":    "code",
    ".ipynb": "code",
    ".txt":   "text",
    ".md":    "text",
    ".vtt":   "oral_video",
    ".srt":   "oral_video",
}


def detect_folder_layout(input_dir: Path) -> str:
    """Return one of: 'flat', 'per_student', 'per_student_per_assignment'.

    Heuristic: find the first supported file in the tree and count its
    depth relative to input_dir. depth=0 -> flat; depth=1 -> per_student;
    depth>=2 -> per_student_per_assignment.
    """
    for p in sorted(input_dir.rglob("*")):
        if not p.is_file():
            continue
        if p.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        depth = len(p.relative_to(input_dir).parts) - 1
        if depth == 0:
            return "flat"
        if depth == 1:
            return "per_student"
        return "per_student_per_assignment"
    return "flat"  # nothing found; default


def parse_ids_from_path(path: Path, input_dir: Path, layout: str) -> Tuple[str, str, str]:
    """Return (student_id, assignment_id, question_id) for one submission file.

    Edit this if your folder layout differs from the three handled below.
    """
    rel = path.relative_to(input_dir)
    parts = rel.parts  # e.g. ('S001', 'A001', 'Q1.py')
    stem = path.stem   # filename without extension

    if layout == "flat":
        # Filename: <student>_<assignment>_<question>.<ext>
        seg = stem.split("_")
        if len(seg) >= 3:
            return seg[0], seg[1], "_".join(seg[2:])
        return "UNKNOWN_STUDENT", "UNKNOWN_ASSIGNMENT", stem

    if layout == "per_student":
        # Folder: student. Filename: <assignment>_<question>.<ext>
        student_id = parts[0]
        seg = stem.split("_")
        if len(seg) >= 2:
            return student_id, seg[0], "_".join(seg[1:])
        return student_id, "UNKNOWN_ASSIGNMENT", stem

    # per_student_per_assignment
    # Folders: student/assignment. Filename: <question>.<ext>
    student_id    = parts[0]
    assignment_id = parts[1] if len(parts) >= 3 else "UNKNOWN_ASSIGNMENT"
    question_id   = stem
    return student_id, assignment_id, question_id


def infer_rubric_type(modality: str, assignment_id: str) -> str:
    """Cheap heuristic from assignment_id; replace with your own mapping if needed."""
    aid = assignment_id.lower()
    if "interview" in aid or "mock" in aid:
        return "mock_interview"
    if "eda" in aid:
        return "open_ended_eda"
    if modality == "oral_video":
        return "mock_interview"
    if modality == "code":
        return "scaffolded_coding"
    return "free_response"


## 13. Load all submissions from Drive

In [ ]:
def load_records_from_folders(input_dir: Path) -> List[FeatureRecord]:
    layout = detect_folder_layout(input_dir)
    print(f"Inferred folder layout: {layout!r}")
    print()

    records: List[FeatureRecord] = []
    skipped: List[Tuple[Path, str]] = []  # (path, reason)

    for p in sorted(input_dir.rglob("*")):
        if not p.is_file():
            continue
        ext = p.suffix.lower()
        if ext not in SUPPORTED_EXTENSIONS:
            if not p.name.startswith("."):  # silently ignore .DS_Store etc.
                skipped.append((p, f"unsupported extension {ext}"))
            continue

        modality = SUPPORTED_EXTENSIONS[ext]
        try:
            student_id, assignment_id, question_id = parse_ids_from_path(p, input_dir, layout)
        except Exception as e:
            skipped.append((p, f"id-parse error: {e}"))
            continue

        text_content: Optional[str]       = None
        code_content: Optional[str]       = None
        transcript_content: Optional[str] = None
        duration_seconds: Optional[float] = None

        try:
            if ext == ".py":
                code_content = read_python_file(p)
            elif ext == ".ipynb":
                code_content = read_ipynb_file(p)
            elif ext in (".txt", ".md"):
                text_content = read_text_file(p)
            elif ext in (".vtt", ".srt"):
                transcript_content, duration_seconds = read_vtt_or_srt(p)
        except Exception as e:
            skipped.append((p, f"read error: {e}"))
            continue

        features = extract_features_for_question(
            modality=modality,
            text=text_content,
            code_text=code_content,
            transcript=transcript_content,
            duration_seconds=duration_seconds,
        )

        rubric_type = infer_rubric_type(modality, assignment_id)

        records.append(FeatureRecord(
            student_id=student_id,
            assignment_id=assignment_id,
            question_id=question_id,
            modality=modality,
            rubric_type=rubric_type,
            features=features,
            # Score fields intentionally left as None — populated downstream
        ))

    print(f"Loaded {len(records)} record(s).")
    if skipped:
        print(f"Skipped {len(skipped)} file(s):")
        for p, reason in skipped[:10]:
            print(f"  {p.relative_to(input_dir)}  ->  {reason}")
        if len(skipped) > 10:
            print(f"  ... ({len(skipped) - 10} more)")
    return records


records = load_records_from_folders(INPUT_DIR)

## 14. Build the feature DataFrame

In [ ]:
if not records:
    raise RuntimeError(
        f"No records were loaded from {INPUT_DIR}. "
        f"Check the folder contents, supported extensions, and the inferred layout above."
    )

df = records_to_dataframe(records)
print(f"DataFrame shape: {df.shape}")
print()
print("Modality breakdown:")
print(df["modality"].value_counts())
print()
print("Per-student record counts (top 10):")
print(df["student_id"].value_counts().head(10))
print()
print("Preview:")
df.head()

## 15. Save the DataFrame to Drive

Writes to `MEng Path Assignments/Regression Analysis Datasets/feature_extraction_output.csv`.

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

written_size = OUTPUT_PATH.stat().st_size
print(f"Saved to: {OUTPUT_PATH}")
print(f"File size: {written_size:,} bytes")
print(f"Rows:      {len(df):,}")
print(f"Columns:   {len(df.columns):,}")
print()
print("ID columns preview:")
print(df[["student_id", "assignment_id", "question_id", "modality", "rubric_type"]].head(10))

## 16. Round-trip check (optional)

Re-reads the saved CSV to confirm it is well-formed and shareable with downstream notebooks.

In [ ]:
roundtrip = pd.read_csv(OUTPUT_PATH)
assert roundtrip.shape == df.shape, (
    f"Shape mismatch on round-trip: read {roundtrip.shape}, expected {df.shape}"
)
print(f"Round-trip OK - shape preserved: {roundtrip.shape}")
print()
print("Score columns are blank as expected:")
for col in ["ai_score_norm", "human_score_norm", "semantic_entropy", "ai_confidence", "disagreement"]:
    n_present = roundtrip[col].notna().sum()
    print(f"  {col:25s} non-null count = {n_present}  (expected 0)")